#### OPTIMIZE Intro

In [0]:
df = spark.read.parquet("abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_201_99457.parquet")
display(df.limit(5))

In [0]:
print(df.count())
print(df.select("category").distinct().count())

In [0]:
df.repartition(5).write.mode("overwrite").partitionBy("category").saveAsTable("deltacatalog.deltadb.optimize_ex1")

In [0]:
%%time
df_ex1 = spark.read.table("deltacatalog.deltadb.optimize_ex1")
df_out = df_ex1.where(df_ex1.category == "Clothing").collect()

In [0]:
from delta.tables import DeltaTable
table = DeltaTable.forName(spark, "deltacatalog.deltadb.optimize_ex1")
table.optimize().executeCompaction()

In [0]:
table.vacuum(0)

In [0]:
%sql 
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

In [0]:
%%time
df_ex1 = spark.read.table("deltacatalog.deltadb.optimize_ex1")
df_out = df_ex1.where(df_ex1.category == "Clothing").collect()

In [0]:
import pyspark.sql.functions as F
df_fruits = df.filter(df.category == "Clothing").withColumn("category", F.lit("Fruits"))

In [0]:
df_fruits.select("category").distinct().count()

In [0]:
df_fruits.repartition(5).write.mode("overwrite").partitionBy("category").saveAsTable("deltacatalog.deltadb.optimize_ex1")

In [0]:
# table.optimize().where("category = 'Fruits'").executeCompaction()

In [0]:
%sql
OPTIMIZE deltacatalog.deltadb.optimize_ex1
WHERE category = 'Fruits';

In [0]:
%sql
VACUUM deltacatalog.deltadb.optimize_ex1 RETAIN 0 HOURS;

#### Optimized Write

In [0]:
df.repartition(288).write.mode("overwrite").partitionBy("category").saveAsTable("deltacatalog.deltadb.optimize_ex2")

In [0]:
%%time
df_ex2 = spark.read.table("deltacatalog.deltadb.optimize_ex2")
df_out = df_ex2.where(df_ex2.category == "Clothing").collect()

In [0]:
df.repartition(288).write.mode("overwrite").partitionBy("category").option("optimizeWrite", "True").saveAsTable("deltacatalog.deltadb.optimize_ex3")

In [0]:
%%time
df_ex3 = spark.read.table("deltacatalog.deltadb.optimize_ex3")
df_out = df_ex3.where(df_ex3.category == "Clothing").collect()

#### Auto Compaction

In [0]:
%sql 
ALTER TABLE deltacatalog.deltadb.optimize_ex3 SET TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'false');
ALTER TABLE deltacatalog.deltadb.optimize_ex3 SET TBLPROPERTIES ('delta.autoOptimize.autoCompact' = 'true');

In [0]:
spark.conf.get("spark.databricks.delta.autoCompact.minNumFiles")
# spark.conf.set("spark.databricks.delta.autoCompact.minNumFiles", "3")

In [0]:
df_detergents = df.filter(df.category == "Clothing").withColumn("category", F.lit("Detergents"))

In [0]:
df_detergents.repartition(5).write.mode("overwrite").partitionBy("category").saveAsTable("deltacatalog.deltadb.optimize_ex3")

#### VACUUM 
(How it limits ability to time travel)

In [0]:
df_101_150 = (
    spark.read.parquet("abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_101_200.parquet")
    .filter(F.col("customer_id").between(101, 150))
)

df_151_200 = (
    spark.read.parquet("abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_101_200.parquet")
    .filter(F.col("customer_id").between(151, 200))
)

In [0]:
df_101_150.write.mode("overwrite").saveAsTable("deltacatalog.deltadb.vacuum_ex1")

In [0]:
%sql
DESCRIBE HISTORY deltacatalog.deltadb.vacuum_ex1;

In [0]:
df_151_200.write.mode("append").saveAsTable("deltacatalog.deltadb.vacuum_ex1")

In [0]:
%sql
DESCRIBE HISTORY deltacatalog.deltadb.vacuum_ex1;

In [0]:
%sql
DELETE FROM deltacatalog.deltadb.vacuum_ex1 WHERE customer_id BETWEEN 151 AND 200; 

In [0]:
%sql
DESCRIBE HISTORY deltacatalog.deltadb.vacuum_ex1;

In [0]:
%sql 
SELECT MIN(customer_id), MAX(customer_id)
FROM deltacatalog.deltadb.vacuum_ex1;

In [0]:
%sql 
SELECT MIN(customer_id), MAX(customer_id)
FROM deltacatalog.deltadb.vacuum_ex1 VERSION AS OF 1;

In [0]:
%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

In [0]:
%sql
VACUUM deltacatalog.deltadb.vacuum_ex1 RETAIN 0 HOURS;

In [0]:
%sql
DESCRIBE HISTORY deltacatalog.deltadb.vacuum_ex1;

In [0]:
%sql
SELECT * 
FROM deltacatalog.deltadb.vacuum_ex1 VERSION AS OF 0;